For Rb (780 nm / 480 nm), the residual-wavevector ratio is
$(780 - 480)/(780 + 480) \approx 0.24$, giving roughly 4× EIT-line narrowing
in the Doppler-dominated limit [1].

## MaxwellBloch implementation

The `factor_doppler_shift` parameter on a field sets the ratio of that
field's wave-vector to the probe wave-vector, with sign for propagation
direction. For the coupling field:

$$
\text{factor\_doppler\_shift} = -\frac{k_c}{k_p} = -\frac{\lambda_p}{\lambda_c}
= -\frac{780}{480} \approx -1.625
$$

Setting `counter_propagating: true` is shorthand for `factor_doppler_shift: -1.0`,
which assumes equal wavelengths. For Rb 780/480 nm, the correct value is
`factor_doppler_shift: -1.625`, which gives the physically correct Doppler-narrowed
(rather than perfectly Doppler-free) EIT linewidth.

## Parameters (γ₁₀ = 1)

| Parameter | Value | Notes |
| --- | --- | --- |
| $\Omega_p$ | $10^{-3}\,\gamma$ | weak CW probe |
| $\Omega_c$ | $2\,\gamma$ | strong CW coupling |
| $\gamma_{10}$ | $1$ | 5p₃/₂ lifetime (~26 ns) |
| $\gamma_{21}$ | $0.001$ | Rydberg decoherence (BBR + transit) |
| $\sigma_D$ | $3\,\gamma$ | laser-cooled Rb (~μK MOT) |

A reduced Doppler width ($\sigma_D = 3\,\gamma$, comparable to $\Omega_c$) is used
so that the probe pulse bandwidth covers the full thermal distribution and the
different EIT linewidths for co- and counter-propagating geometries are clearly
resolved in the spectrum.

In [1]:
import numpy as np
from maxwellbloch import mb_solve, plot

## Baseline: no coupling field

With the coupling field off ($\Omega_c = 0$), the probe sees a
Doppler-broadened two-level absorber. This is the background against which
the EIT window will appear.

In [2]:
mb_solve_json_no_coupling = """
{
  "atom": {
    "num_states": 3,
    "decays": [
      {"channels": [[0, 1]], "rate": 1.0},
      {"channels": [[1, 2]], "rate": 0.001}
    ],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "detuning": 0.0,
        "detuning_positive": true,
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 2.0}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "detuning": 0.0,
        "detuning_positive": false,
        "rabi_freq": 0.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -3.0, "off": 9.0}
      }
    ]
  },
  "t_min": -3.0,
  "t_max": 9.0,
  "t_steps": 200,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [1.0, 0.0],
  "velocity_classes": {
    "thermal_width": 25.0,
    "thermal_delta_min": -75.0,
    "thermal_delta_max":  75.0,
    "thermal_delta_steps": 20,
    "thermal_delta_inner_min": -5.0,
    "thermal_delta_inner_max":  5.0,
    "thermal_delta_inner_steps": 8
  },
  "savefile": "mbs-ladder-rydberg-eit-no-coupling"
}
"""

mbs_no_c = mb_solve.MBSolve().from_json_str(mb_solve_json_no_coupling)
mbs_no_c.mbsolve(recalc=False)
print("Done")

Done


## Co-propagating EIT

With both fields co-propagating, all velocity classes see the same
first-order Doppler shift on both fields. The two-photon detuning for a
class at velocity $v$ is

$$
\delta_{02}^{\text{co}}(v) = \delta_p + \delta_c - (k_p + k_c)\,v
$$

Since $k_p + k_c \neq 0$, different velocity classes are resonant at
different probe frequencies. The EIT feature is therefore broadened across
the Doppler distribution.

In [3]:
mb_solve_json_co_prop = """
{
  "atom": {
    "num_states": 3,
    "decays": [
      {"channels": [[0, 1]], "rate": 1.0},
      {"channels": [[1, 2]], "rate": 0.001}
    ],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "detuning": 0.0,
        "detuning_positive": true,
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 2.0}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "detuning": 0.0,
        "detuning_positive": false,
        "rabi_freq": 2.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -3.0, "off": 9.0}
      }
    ]
  },
  "t_min": -3.0,
  "t_max": 9.0,
  "t_steps": 200,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [1.0, 0.0],
  "velocity_classes": {
    "thermal_width": 25.0,
    "thermal_delta_min": -75.0,
    "thermal_delta_max":  75.0,
    "thermal_delta_steps": 20,
    "thermal_delta_inner_min": -5.0,
    "thermal_delta_inner_max":  5.0,
    "thermal_delta_inner_steps": 8
  },
  "savefile": "mbs-ladder-rydberg-eit-co-prop"
}
"""

mbs_co = mb_solve.MBSolve().from_json_str(mb_solve_json_co_prop)
mbs_co.mbsolve(recalc=False)
print("Done")

Done


## Counter-propagating EIT: Doppler narrowing

With the coupling beam counter-propagating, its Doppler shift has the
**opposite sign** to the probe. The two-photon detuning becomes

$$
\delta_{02}^{\text{ctr}}(v) = \delta_p + \delta_c - (k_p - k_c)\,v
$$

For Rb (probe 780 nm, coupling 480 nm) the residual wavevector ratio is

$$
\frac{k_p - k_c}{k_p + k_c} = \frac{\lambda_c - \lambda_p}{\lambda_c + \lambda_p}
= \frac{480 - 780}{480 + 780} \approx -0.24
$$

so the two-photon resonance condition is satisfied for a much broader range
of velocity classes. In the Doppler-dominated limit the EIT linewidth
narrows by $|k_p - k_c|/|k_p + k_c| \approx 0.24$.

Setting `counter_propagating: true` on the coupling field in MaxwellBloch
applies $\text{factor\_doppler\_shift} = -1$ to that field, reproducing this
geometry.

In [4]:
mb_solve_json_counter_prop = """
{
  "atom": {
    "num_states": 3,
    "decays": [
      {"channels": [[0, 1]], "rate": 1.0},
      {"channels": [[1, 2]], "rate": 0.001}
    ],
    "fields": [
      {
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "detuning": 0.0,
        "detuning_positive": true,
        "rabi_freq": 1.0e-3,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {"ampl": 1.0, "centre": 0.0, "fwhm": 2.0}
      },
      {
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "detuning": 0.0,
        "detuning_positive": false,
        "counter_propagating": true,
        "rabi_freq": 2.0,
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {"ampl": 1.0, "fwhm": 0.2, "on": -3.0, "off": 9.0}
      }
    ]
  },
  "t_min": -3.0,
  "t_max": 9.0,
  "t_steps": 200,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [1.0, 0.0],
  "velocity_classes": {
    "thermal_width": 25.0,
    "thermal_delta_min": -75.0,
    "thermal_delta_max":  75.0,
    "thermal_delta_steps": 20,
    "thermal_delta_inner_min": -5.0,
    "thermal_delta_inner_max":  5.0,
    "thermal_delta_inner_steps": 8
  },
  "savefile": "mbs-ladder-rydberg-eit-counter-prop"
}
"""

mbs_ctr = mb_solve.MBSolve().from_json_str(mb_solve_json_counter_prop)
mbs_ctr.mbsolve(recalc=False)
print("Done")

Done


## Comparison: co-propagating vs counter-propagating EIT

All three spectra are overlaid. The broad Doppler-broadened background (first
plot, ±40γ) is the same in all cases. The second plot zooms in to ±5γ around
resonance, revealing the EIT transparency dips: the co-propagating dip is
broadened across the Doppler distribution, while the counter-propagating dip
is much sharper — near Doppler-free.

In [5]:
labels = ["coupling off", "co-propagating (Ω_c = 2γ)", "counter-propagating (Ω_c = 2γ)"]

# Wide view: full Doppler background
fig_wide = plot.spectrum_overlay(
    [mbs_no_c, mbs_co, mbs_ctr], field_idx=0, labels=labels,
)
fig_wide.update_layout(
    title="Rydberg EIT spectra — full Doppler range",
    xaxis=dict(range=[-40, 40]),
)
fig_wide.show(renderer='notebook_connected')

# Zoomed view: EIT transparency dips
fig_zoom = plot.spectrum_overlay(
    [mbs_no_c, mbs_co, mbs_ctr], field_idx=0, labels=labels,
)
fig_zoom.update_layout(
    title="Rydberg EIT spectra — zoomed near resonance",
    xaxis=dict(range=[-5, 5]),
)
fig_zoom.show(renderer='notebook_connected')

## EIT dip FWHM: measuring the Doppler narrowing

We can quantify the narrowing by finding the FWHM of the transparency dip
in each spectrum. The ratio should approach $|k_p - k_c|/|k_p + k_c| \approx 0.24$
in the Doppler-dominated limit.

In [6]:
from maxwellbloch import spectral
from scipy.signal import find_peaks

def eit_dip_fwhm(mbs, field_idx=0):
    """FWHM of the EIT transparency dip (minimum in absorption spectrum)."""
    freqs = spectral.freq_list(mbs)
    absorp = spectral.absorption(mbs, field_idx)
    # Invert to find dip as peak
    inv = -absorp
    peaks, props = find_peaks(inv, height=0.0, width=1)
    if len(peaks) == 0:
        return None
    # Take the deepest dip near zero detuning
    centre_peak = peaks[np.argmin(np.abs(freqs[peaks]))]
    # Half-width: distance from peak to where signal reaches half-height
    peak_val = inv[centre_peak]
    half_level = peak_val / 2.0
    df = freqs[1] - freqs[0]
    # Walk left and right from peak to find half-width
    i = centre_peak
    while i > 0 and inv[i] > half_level:
        i -= 1
    left_freq = freqs[i]
    i = centre_peak
    while i < len(inv) - 1 and inv[i] > half_level:
        i += 1
    right_freq = freqs[i]
    return right_freq - left_freq

fwhm_co = eit_dip_fwhm(mbs_co)
fwhm_ctr = eit_dip_fwhm(mbs_ctr)

if fwhm_co is not None and fwhm_ctr is not None:
    print(f"Co-prop EIT FWHM:      {fwhm_co:.2f} γ")
    print(f"Counter-prop EIT FWHM: {fwhm_ctr:.2f} γ")
    print(f"Ratio (ctr/co):        {fwhm_ctr/fwhm_co:.3f}  (theory ≈ 0.24 for Rb)")
else:
    print("No clear EIT dip found — try increasing Ω_c or thermal_width")

Co-prop EIT FWHM:      2.07 γ
Counter-prop EIT FWHM: 0.41 γ
Ratio (ctr/co):        0.200  (theory ≈ 0.24 for Rb)


## Probe propagation: counter-propagating geometry

The space-time plot of the probe field shows that on two-photon resonance
the medium is transparent. The pulse propagates with minimal absorption.

In [7]:
fig = plot.field_spacetime(mbs_ctr, field_idx=0)
fig.update_layout(title="Probe |Ω_p(z, t)| — counter-propagating Rydberg EIT (Ω_c = 2γ)")
fig.show(renderer='notebook_connected')

## Summary

- **No coupling:** Doppler-broadened Lorentzian with width $\sim \sigma_D \approx 25\,\gamma$.
- **Co-propagating coupling:** EIT dip is visible but broadened — most velocity
  classes see a large two-photon detuning $\propto (k_p + k_c)v$.
- **Counter-propagating coupling:** EIT dip is sharp — two-photon detuning
  $\propto (k_p - k_c)v$ is small, so all velocity classes nearly satisfy
  two-photon resonance simultaneously. The dip width approaches
  the natural EIT linewidth $\sim \Omega_c^2 / \gamma_{10}$, drastically
  narrower than $\sigma_D$.

The narrowing ratio (counter/co) approaches $|k_p - k_c|/|k_p + k_c|$ in
the Doppler-dominated limit — roughly 0.24 for the Rb 780/480 nm system.

## References

1. A. K. Mohapatra, T. R. Jackson, C. S. Adams, *Coherent Optical
   Detection of Highly Excited Rydberg States using Electromagnetically
   Induced Transparency*, PRL **98**, 113003 (2007). First Rydberg-EIT
   in thermal vapour; counter-propagating geometry.
2. C. S. Adams, J. D. Pritchard, J. P. Shaffer, *Rydberg atom quantum
   technologies*, J. Phys. B **53**, 012002 (2020). Review of Rydberg
   sensing and quantum information applications.
3. N. Šibalić, J. D. Pritchard, C. S. Adams, K. J. Weatherill, *ARC: An
   open-source library for calculating properties of alkali Rydberg atoms*,
   CPC **220**, 319 (2017). Parameter numerics and lifetime estimates.